# Preliminary SAE results: reconstruction, sparsity, and dictionary coverage

This comparison asks how two finalized sparse autoencoders (SAEs) balance **reconstruction quality**, **sparsity per example**, and **dictionary coverage across validation examples**. All four metrics come from evaluation of each model's saved best checkpoint on the same validation activation data. These are descriptive results for these two checkpoints, not averages across repeated seeds.

Both models use layer 33 activations with 2,560 input dimensions and a dictionary of 12,888 latent features. Inputs are divided by a single RMS scale computed from training activations; that scale is reused for validation. The table compares reconstruction error in these normalized units.

## Models and saved sources

- **TopK + AuxK:** TopK sparsity with `top_k = 128`, plus a training-only auxiliary reconstruction objective with `aux_coefficient = 0.0078125`. AuxK targets eligible inactive features during training. The finalized model is trial 3 of the AuxK diagnostics study; its best checkpoint is epoch 179.
  - Run: `results/sae/musique/runs/layer_33/20261013T233444019723Z-e5943205b957/`
- **L1:** an L1 penalty on latent activations with `sparsity_coefficient = 5.0`. Its number of positive activations varies by example. The finalized model is trial 3 of the L1 refinement study; its best checkpoint is epoch 111. This was the coverage-oriented choice; the study's automatic activity-budget selection suggested trial 7 instead.
  - Run: `results/sae/musique/runs/layer_33/20261014T161949048498Z-edfc89a42125/`

The code reads `config.json` for model settings, `manifest.json` for validation-data provenance, and `history.json` → `summary` → `frozen_checkpoint_metrics` → `validation` for metrics. These frozen metrics describe the checkpoint used for inference; the last training-history row need not describe that checkpoint.

## What each metric means

| Metric | Definition | Interpretation |
| --- | --- | --- |
| Reconstruction MSE | Mean squared difference between the original and reconstructed activations, averaged over validation examples and input dimensions, in RMS-normalized units. Saved as `rec_loss`. | Lower means a more faithful reconstruction. We compare reconstruction alone because the models' total losses contain different penalties. |
| Variance explained | $1 - \mathrm{SSE}_{\mathrm{reconstruction}} / \mathrm{SSE}_{\mathrm{baseline}}$, where the baseline predicts the **training-mean activation vector** for each validation example. Saved as `variance_explained`. | Higher means more improvement over this constant baseline; 100% is perfect reconstruction, 0% matches the baseline, and negative values are worse. |
| Mean active features per example | Average count of latent features with a **positive** activation on a validation example. Saved as `mean_active_features`. | Lower means a sparser representation per example. This is an observed count, not simply the configured TopK value. |
| Dictionary features active at least once on validation | $100\times(1-\mathrm{inactive\_feature\_fraction})$. A feature counts as used if it has a positive activation on at least one validation example. | Higher means broader coverage of the dictionary across this split. It does not mean more features are active simultaneously. |

## How to interpret the comparison

For these checkpoints, TopK + AuxK reconstructs better (MSE approximately 0.0434 versus 0.0589; variance explained 83.2% versus 77.2%). L1 activates fewer features per example (approximately 66 versus 128) while using more of the dictionary across validation examples (97.3% versus 24.2%). Sparse individual representations and broad aggregate dictionary coverage can therefore coexist.

This tradeoff motivates carrying both models into downstream experiments. Coverage does **not** establish semantic interpretability, useful causal features, or successful interventions. A feature never observed on validation is also not necessarily permanently dead: split-level inactivity differs from the training-time persistent-inactivity diagnostic. The comparison does not isolate the effect of one hyperparameter, since these configurations differ in multiple settings.

## Reproduce the table

Run the following cells from either the repository root or the `cookbooks` directory. No training or checkpoint loading is required. Run IDs are explicit so future experiments do not silently change which models are compared. The loader checks that the recorded validation hashes, layer, dimensions, and training RMS scale agree. Values retain full precision in `comparison_numeric`; only the displayed table is rounded.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the repository without depending on the notebook's working directory.
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "results/sae/musique/runs/layer_33").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project repository.")

RUN_IDS = {
    "TopK + AuxK": "20261013T233444019723Z-e5943205b957",
    "L1": "20261014T161949048498Z-edfc89a42125",
}

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

runs = {}
for label, run_id in RUN_IDS.items():
    directory = REPO_ROOT / "results/sae/musique/runs/layer_33" / run_id
    config = read_json(directory / "config.json")
    manifest = read_json(directory / "manifest.json")
    history = read_json(directory / "history.json")
    if manifest["status"] != "completed":
        raise ValueError(f"{label}: expected a completed run.")
    runs[label] = {
        "config": config,
        "manifest": manifest,
        "summary": history["summary"],
        "metrics": history["summary"]["frozen_checkpoint_metrics"]["validation"],
    }

validation_hashes = {
    run["manifest"]["activation_files"]["validation"]["sha256"]
    for run in runs.values()
}
if len(validation_hashes) != 1 or not all(validation_hashes):
    raise ValueError("The runs do not record matching validation activation hashes.")
for field in ("layer", "input_dim", "latent_dim", "activation_rms_scale"):
    if len({run["config"][field] for run in runs.values()}) != 1:
        raise ValueError(f"The runs differ in {field}; review comparability first.")


In [ ]:
# Percent-valued metrics stay as fractions here to retain their natural units.
comparison_numeric = pd.DataFrame({
    label: {
        "Reconstruction MSE": run["metrics"]["rec_loss"],
        "Variance explained": run["metrics"]["variance_explained"],
        "Mean active features per example": run["metrics"]["mean_active_features"],
        "Dictionary features active at least once on validation": (
            1.0 - run["metrics"]["inactive_feature_fraction"]
        ),
    }
    for label, run in runs.items()
})
comparison_numeric.index.name = "Metric"

row_formats = {
    "Reconstruction MSE": ".4f",
    "Variance explained": ".1%",
    "Mean active features per example": ".1f",
    "Dictionary features active at least once on validation": ".1%",
}
comparison_table = pd.DataFrame(
    {
        label: [format(comparison_numeric.loc[metric, label], row_formats[metric])
                for metric in comparison_numeric.index]
        for label in comparison_numeric.columns
    },
    index=comparison_numeric.index,
)
display(comparison_table)


## Export the comparison table for slides

Save the displayed table to `reports/figures/sae_preliminary_results/sae_validation_comparison.svg`. This keeps presentation figures together under `reports`, separate from experiment artifacts. The directory is created automatically, and rerunning the cell updates the same SVG. The export uses the values in `comparison_table`, with a title and validation-checkpoint caption. SVG preserves sharp text and lines when resized; text is kept editable for slide or vector editors.


In [ ]:
import matplotlib.pyplot as plt

FIGURE_DIR = REPO_ROOT / "reports" / "figures" / "sae_preliminary_results"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
svg_path = FIGURE_DIR / "sae_validation_comparison.svg"

# Keep SVG labels as editable text rather than converting them to paths.
with plt.rc_context({"svg.fonttype": "none", "font.family": "DejaVu Sans"}):
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.axis("off")
    fig.suptitle(
        "SAE validation comparison", x=0.06, y=0.96,
        ha="left", fontsize=18, fontweight="bold", color="#172B4D",
    )
    rows = [
        [metric, *comparison_table.loc[metric].tolist()]
        for metric in comparison_table.index
    ]
    table = ax.table(
        cellText=rows,
        colLabels=["Metric", *comparison_table.columns.tolist()],
        colWidths=[0.70, 0.15, 0.15],
        cellLoc="center", bbox=[0, 0.13, 1, 0.83],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    for (row, column), cell in table.get_celld().items():
        cell.set_edgecolor("#D9E2EC")
        cell.set_linewidth(0.7)
        cell.PAD = 0.025
        if row == 0:
            cell.set_facecolor("#172B4D")
            cell.set_text_props(color="white", weight="bold")
        else:
            cell.set_facecolor("#F0F4F8" if row % 2 else "white")
            cell.set_text_props(color="#172B4D")
        if column == 0:
            cell.set_text_props(ha="left")
    fig.text(
        0.06, 0.055,
        "Saved best checkpoints · Layer 33 · Validation split · 12,888 latent features",
        fontsize=10, color="#52606D",
    )
    fig.subplots_adjust(left=0.06, right=0.97, top=0.84, bottom=0.12)
    fig.savefig(svg_path, format="svg", bbox_inches="tight", facecolor="white")
    plt.close(fig)

print(f"Saved SVG: {svg_path.relative_to(REPO_ROOT)}")
